# 第12回　ベイズ統計の入り口
## ―― データで「信念」を更新する、もう一つの確率

統計学Ⅰ（B）　／　北星学園大学

注目は ――

> 確率は「長い目で見た頻度」だけではない。「**今どれくらい確からしいか（信念の度合い）**」でもある。

### フック：99%正確な検査で陽性。あなたは病気？

> ある病気の検査は **99%正確**（病気の人の99%が陽性、健康な人の99%が陰性）。
> あなたは陽性だった。**本当に病気である確率は？**
>
> 直感は「99%だ、もうダメだ…」と言う。本当だろうか？

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats
print("準備OK")

In [ ]:
# 10000人で考える（有病率1%＝100人が病気）
有病率, 感度, 特異度 = 0.01, 0.99, 0.99
病気 = 10000*有病率
健康 = 10000-病気
病気で陽性 = 病気*感度          # 100人中99人
健康で陽性 = 健康*(1-特異度)     # 9900人中99人（偽陽性！）
print(f"病気の人 {病気:.0f}人 → うち陽性 {病気で陽性:.0f}人")
print(f"健康な人 {健康:.0f}人 → うち陽性（偽陽性）{健康で陽性:.0f}人")
陽性合計 = 病気で陽性 + 健康で陽性
print(f"\n陽性になった人 合計 {陽性合計:.0f}人。そのうち本当に病気は {病気で陽性:.0f}人。")
print(f"→ 陽性なら本当に病気の確率 = {病気で陽性/陽性合計*100:.0f}%（直感の99%ではなく、たった50%！）")

**陽性でも、本当に病気の確率は50%。** なぜか。病気の人は元々少ない（1%）。だから「健康なのに偶然陽性（偽陽性）」の人数が、「本当に病気で陽性」の人数と同じくらい出てしまう。

直感が外れたのは、**事前の情報（有病率1%）を無視した**から。これを取り込むのが **ベイズの考え方**だ。

$$ P(\text{病気}\mid\text{陽性}) = \frac{P(\text{陽性}\mid\text{病気})\,P(\text{病気})}{P(\text{陽性})} $$

- $P(\text{病気})$＝**事前確率**（検査前の信念。ここでは有病率1%）
- $P(\text{陽性}\mid\text{病気})$＝**尤度**（病気なら陽性が出やすさ）
- $P(\text{病気}\mid\text{陽性})$＝**事後確率**（検査後に更新された信念）

---
## 1. 頻度論 vs ベイズ ―― 2つの世界観

| | 頻度論（これまで） | ベイズ |
|---|---|---|
| 確率とは | 長い目で見た**頻度**（手続きの当たり率） | **信念の度合い**（今どれくらい確からしいか） |
| パラメータ（母平均など） | **固定された定数**（第7回） | **確率分布**を持つ（信念として） |
| やること | データから手続きで推定 | **事前 × データ → 事後** に信念を更新 |
| 95%区間 | 信頼区間（手続きの当たり率） | 信用区間（真値がこの中にある確率95%、と言える） |

どちらが正しいではなく、**世界の捉え方が違う**。

---
## 2. ベイズ更新を見る ―― コインの表が出る確率θ

あるコインの「表が出る確率 θ」を知りたい。最初は何も分からないので、θは0〜1のどれも同じくらい、という**一様な事前**から始める。

コインを投げるたびに、その結果（データ）で θ の分布（信念）を更新していく。データが増えると、分布はどうなる？

（数学的に、表/裏の観測は **ベータ分布**で事後がきれいに計算できる：表k回・裏(n−k)回を見たら 事後は Beta(1+k, 1+n−k)。）

In [ ]:
rng = np.random.default_rng(2026)
真のθ = 0.7
flips = rng.random(200) < 真のθ      # True=表

x = np.linspace(0, 1, 300)
plt.figure(figsize=(7.5, 4))
for n in [0, 5, 20, 100]:
    k = int(flips[:n].sum())
    a, b = 1+k, 1+(n-k)
    plt.plot(x, stats.beta.pdf(x, a, b), lw=2, label=f"n={n}（表{k}回）事後平均{a/(a+b):.2f}")
plt.axvline(真のθ, color="gray", ls="--", label="真の値 0.7")
plt.xlabel("θ（表が出る確率）への信念"); plt.ylabel("確からしさ")
plt.title("データが増えるほど、信念は真値0.7の周りに締まる"); plt.legend(); plt.show()

In [ ]:
print("投げた回数 / 事後平均 / 95%信用区間（この中にθがある確率95%）")
for n in [0, 5, 20, 100]:
    k = int(flips[:n].sum()); a, b = 1+k, 1+(n-k)
    lo, hi = stats.beta.ppf([.025, .975], a, b)
    print(f"  n={n:3d}: 平均{a/(a+b):.2f}  区間[{lo:.2f}, {hi:.2f}]（幅{hi-lo:.2f}）")
print("\nデータが増えるほど区間は狭まり、真値0.7に近づく。")

n=0では「全く分からない」（幅0.95）。投げるほど信念は締まり、n=100では真値0.7のすぐ近く（幅0.17）まで絞り込めた。**これがベイズ更新：データを見るたびに信念を上書きしていく。**

---
## 3. 「事前は主観だから非科学的」？

よくある批判。だが――**事前が違っても、データが増えれば事後はほぼ一致する**。確かめよう。同じ100回の結果を、正反対の事前を持つ2人が見る。

In [ ]:
k = int(flips[:100].sum())   # 100回中の表の回数
print(f"100回中 表{k}回 を、正反対の事前を持つ2人が見ると…\n")
for name, (a0, b0) in {"懐疑的な人 Beta(2,8)": (2,8), "楽観的な人 Beta(8,2)": (8,2)}.items():
    a, b = a0+k, b0+(100-k)
    print(f"  {name}: 事前平均 {a0/(a0+b0):.2f} → 事後平均 {a/(a+b):.2f}")
print("\n出発点（信念）が正反対でも、データを見たあとの結論はほぼ同じ（約0.70〜0.76）。")
print("→ データが十分あれば、事前の影響は薄まる。事前は『非科学』ではなく『出発点』。")

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| ベイズの確率 | 信念の度合い（今どれくらい確からしいか） |
| 事前→尤度→事後 | 事前の信念を、データ（尤度）で更新して事後にする |
| ベイズ更新 | データを見るたびに信念を上書き。増えるほど締まる |
| 信用区間 | 「真値がこの中にある確率95%」と**言える**（頻度論の信頼区間とは違う・第7回） |
| ❌ 誤り | 事前は主観だから非科学／ベイズは頻度論の上位互換 |

> **ベイズは『データで信念を更新する』枠組み。頻度論とは世界観が違うだけで、優劣ではない。**
> 検査の例のように、**事前情報を無視すると直感は大きく外れる**。

**課題（Moodle）**：頻度論とベイズの違いを自分の言葉で／陽性的中率を計算する。